# Smart Shelf CV — Fine-tune YOLOv8

**Role:** AI Eng 1 (Object Detection)

Dataset: 30 รูป, 7 class (`Bottle`, `Box`, `Carton`, `Jar`, `Pouch`, `Tube`, `can`) จาก Roboflow (SKU-110k subset ที่ relabel แล้ว)

ทำตามทีละ cell ตามลำดับ ห้ามข้าม step

## Step 1: เช็ค GPU

ก่อนรัน cell นี้ ไปที่เมนู **Runtime > Change runtime type > Hardware accelerator > GPU** ก่อน ไม่งั้นเทรนจะช้ามาก

รัน cell ด้านล่างแล้วต้องเห็นชื่อ GPU (เช่น Tesla T4) ถ้าขึ้น error แปลว่ายังไม่ได้เปิด GPU

In [ ]:
!nvidia-smi

## Step 2: ติดตั้ง ultralytics (YOLOv8)

In [ ]:
!pip install ultralytics -q

import ultralytics
ultralytics.checks()

## Step 3: อัปโหลด dataset

ก่อนรัน: ที่เครื่องตัวเอง ให้ zip โฟลเดอร์ `train/`, `valid/`, `test/`, และไฟล์ `data.yaml` รวมเป็นไฟล์เดียว เช่น `dataset.zip`
(ต้อง zip จากข้างในโฟลเดอร์ ให้ `data.yaml` อยู่ระดับบนสุดของ zip ไม่ใช่ซ้อนโฟลเดอร์อีกชั้น)

รัน cell ด้านล่างแล้วเลือกไฟล์ `dataset.zip` ที่เตรียมไว้

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import zipfile

zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as zip_ref:
    zip_ref.extractall('dataset')

!ls dataset

## Step 4: เช็คไฟล์ data.yaml

ต้องเห็น class ครบ 7 อัน และ path ของ train/val/test ถูกต้อง

In [ ]:
!cat dataset/data.yaml

**หมายเหตุ:** ถ้า path ใน `data.yaml` เป็น `../train/images` แบบ relative path ที่อ้างอิงจากตอน export บนเครื่องเดิม อาจต้องแก้เป็น path เต็มใน Colab เช่น `/content/dataset/train/images` — ถ้า cell train ข้างล่าง error เรื่องหา path ไม่เจอ ให้กลับมาแก้ไฟล์นี้ก่อน

## Step 4.5: อัปโหลด pretrained weight (ถ้าจะลองใช้ YOLO Waste Detection แทน yolov8n.pt)

ไปที่หน้า model บน Roboflow Universe ("YOLO Waste Detection" หรือโมเดล waste/packaging ตัวอื่นที่ใกล้เคียง) เช็คว่ามีไฟล์ `.pt` ให้ดาวน์โหลดไหม (ต้องเป็น **YOLOv8** เท่านั้น ถ้าเป็น version อื่นโหลดเข้า `ultralytics` ไม่ได้)

ดาวน์โหลดมาไว้ที่เครื่องก่อน แล้วรัน cell ด้านล่างเพื่ออัปโหลดเข้า Colab — **ถ้าหาไฟล์ไม่เจอหรือดาวน์โหลดไม่ได้ กด Cancel ตอนอัปโหลดได้เลย ระบบจะ fallback ไปใช้ `yolov8n.pt` ตามปกติ**

In [ ]:
from google.colab import files

print('เลือกไฟล์ .pt ที่ดาวน์โหลดมา หรือกด Cancel เพื่อข้ามและใช้ yolov8n.pt แทน')
uploaded_weight = files.upload()

if uploaded_weight:
    PRETRAINED_WEIGHT = list(uploaded_weight.keys())[0]
else:
    PRETRAINED_WEIGHT = 'yolov8n.pt'

print(f'จะใช้ weight เริ่มต้น: {PRETRAINED_WEIGHT}')

## Step 5: เทรนโมเดล (fine-tune จาก `PRETRAINED_WEIGHT`)

**หมายเหตุสำคัญ:** ชุดข้อมูลตอนนี้มี valid set แค่ 1 รูป ซึ่งน้อยมาก ค่า mAP ที่ได้จะแกว่ง/ไม่น่าเชื่อถือเท่าไหร่ — ใช้เป็นตัวดูแนวโน้มคร่าวๆ ไปก่อนสำหรับรอบทดลองแรกได้ ถ้ามีเวลาค่อยไปเพิ่มรูปใน valid set ทีหลัง

**เพิ่มจากรอบก่อนหน้า:**
- **จุดเริ่มเทรน** — ใช้ `PRETRAINED_WEIGHT` จาก step 4.5 (fallback เป็น `yolov8n.pt` ถ้าไม่ได้อัปโหลด/โหลดไม่ได้)
- **epochs 30 → 80**
- **augmentation** — `hsv_h/hsv_s/hsv_v` (แสง), `degrees`+`scale` (มุมกล้อง/ระยะ), `fliplr=0.5`, `flipud=0.0` (ปิด เพราะสินค้าต้องตั้งตรงเสมอ)
- **`freeze=10` (ใหม่)** — ล็อกน้ำหนักของ 10 ชั้นแรกของ backbone ไม่ให้เปลี่ยนระหว่างเทรน ชั้นแรกๆ ของโมเดลมักเรียนรู้ลักษณะพื้นฐาน (ขอบ, รูปทรง, พื้นผิว) จาก pretrained มาก่อนแล้วซึ่งยังมีประโยชน์อยู่ — ถ้าปล่อยให้เทรนทับทั้งหมดด้วย dataset เล็กแค่ 30 รูป เสี่ยง overfit และ "ลืม" ความรู้เดิมเร็วเกินไป freeze ช่วยให้เทรนเฉพาะชั้นหลังๆ ที่ต้องปรับให้เข้ากับ class ใหม่ ซึ่งเสถียรกว่า

**ถ้า cell ด้านล่าง error เรื่อง architecture/class ไม่ตรงกัน** ให้กลับไป step 4.5 รันใหม่แล้วกด Cancel ตอนอัปโหลด เพื่อ fallback เป็น `yolov8n.pt`

บันทึกผลเป็น `train3` แยกจากรอบก่อนหน้า เพื่อเทียบกันได้ว่า freeze ช่วยจริงไหม

In [ ]:
from ultralytics import YOLO

model = YOLO(PRETRAINED_WEIGHT)

results = model.train(
    data='dataset/data.yaml',
    epochs=80,
    imgsz=640,
    batch=8,
    project='smart_shelf_cv',
    name='train3',
    freeze=10,
    hsv_h=0.02,
    hsv_s=0.6,
    hsv_v=0.4,
    degrees=10,
    scale=0.3,
    fliplr=0.5,
    flipud=0.0,
    mosaic=1.0
)

## Step 6: ดูผลลัพธ์การเทรน (mAP, confusion matrix)

In [ ]:
from IPython.display import Image, display

display(Image(filename='smart_shelf_cv/train3/results.png'))

In [ ]:
display(Image(filename='smart_shelf_cv/train3/confusion_matrix.png'))

## Step 7: ทดสอบ inference กับรูปจาก test set

In [ ]:
import glob

test_images = glob.glob('dataset/test/images/*')
print(f'เจอรูปทดสอบ {len(test_images)} รูป')

predict_results = model.predict(source=test_images[0], save=True, conf=0.25)

In [ ]:
predicted_path = predict_results[0].save_dir
import os

predicted_file = os.path.join(predicted_path, os.path.basename(test_images[0]))
display(Image(filename=predicted_file))

## Step 8: ดาวน์โหลด weight ที่เทรนเสร็จ

ไฟล์ `best.pt` คือโมเดลที่เทรนดีที่สุด — เอาไฟล์นี้ส่งต่อให้ Software Eng ไปเชื่อม endpoint (ตาม tracker Day 4)

In [ ]:
files.download('smart_shelf_cv/train3/weights/best.pt')

## Step 9: ดาวน์โหลดทั้งโฟลเดอร์ run (weights + กราฟผล + log ทั้งหมด)

เผื่อไว้เป็นหลักฐาน/backup หรือจะเอาไปดูผลย้อนหลังทีหลัง — zip ทั้งโฟลเดอร์ `smart_shelf_cv/train3` แล้วโหลดออกมาทีเดียว

**หมายเหตุ:** Colab จะลบไฟล์ทั้งหมดทิ้งถ้า runtime หลุด/timeout ควรดาวน์โหลดเก็บไว้ทันทีหลังเทรนเสร็จ อย่ารอ

In [ ]:
import shutil

shutil.make_archive('train3_results', 'zip', 'smart_shelf_cv/train3')
files.download('train3_results.zip')